<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F%206.7.%20%D0%A1%D1%83%D0%BF%D0%B5%D1%80%D0%B2%D0%B0%D0%B9%D0%B7%D0%B5%D1%80%20%D0%B8%20%D1%81%D0%BB%D0%BE%D0%B6%D0%BD%D0%B0%D1%8F%20%D0%BE%D1%80%D0%BA%D0%B5%D1%81%D1%82%D1%80%D0%B0%D1%86%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.7. Супервайзер и сложная оркестрация

## Введение: от команды равных к иерархии с управляющим

В Лекции 6.6 мы построили multi‑agent систему, где исследователь, критик и писатель работали в цикле, обмениваясь сообщениями и уточняя результаты. Это была **команда равных** – каждый агент выполнял свою роль, но никто не управлял процессом. Мы жёстко задали порядок: исследователь → критик → (возврат) → писатель. Это работало, но было негибко.

Представьте, что вы – руководитель проекта. Вы не просто даёте задание и ждёте результат. Вы оцениваете промежуточные итоги, перераспределяете задачи, меняете приоритеты, подключаете новых экспертов, если это нужно. Именно так работает **супервайзер** – управляющий агент, который координирует работу других агентов, ставит им задачи, собирает результаты и принимает решения о следующих шагах.

В этой лекции мы добавим в нашу multi‑agent систему **супервайзера**. Он будет не просто передавать управление по кругу, а **динамически** решать, какого агента вызвать, когда завершить работу и какие дополнительные действия предпринять. Это сделает систему по‑настоящему гибкой, способной адаптироваться к неожиданным результатам и сложным сценариям.

Мы реализуем супервайзера как отдельного LLM-агента, который видит всю историю диалога, знает, какие агенты доступны, и использует их как «инструменты» для достижения цели. В финале мы увидим, как супервайзер может перепланировать работу на лету, если первоначальный план не сработал.

---

## Тема 1. Зачем нужен супервайзер

### 1.1. Отличие от простой цепочки

В предыдущих лекциях мы использовали **жёсткие цепочки** и **условные переходы**. Например, в Лекции 6.6 мы зафиксировали порядок: исследователь → критик → (если нужно, снова исследователь) → писатель. Это работает для многих задач, но имеет ограничения:

- **Нет выбора** – мы не можем в зависимости от промежуточного результата вызвать другого агента (например, аналитика вместо исследователя).
- **Нет параллельности** – мы не можем запустить двух агентов одновременно и объединить их результаты.
- **Нет стратегического планирования** – мы не можем сказать: «Сначала собери факты, потом проверь их, а если найдешь противоречия – поищи дополнительную информацию в другом источнике».

Супервайзер решает эти проблемы. Он видит всю картину и может перепланировать. Вместо фиксированного графа он принимает решения на каждом шаге:

1. **Какой агент нужен сейчас?** – Исследователь, критик, писатель или кто‑то ещё.
2. **Какую задачу ему дать?** – Не просто «собери факты», а «найди информацию о компании X за 2023 год».
3. **Когда остановиться?** – Если цель достигнута, завершить работу.

Это похоже на то, как менеджер проекта распределяет задачи между разработчиками, тестировщиками и аналитиками, корректируя план по ходу дела.

### 1.2. Примеры использования супервайзера

Супервайзер особенно полезен в сценариях, где:

- **Автоматизация бизнес-процессов** – обработка заявок, генерация отчётов, взаимодействие с клиентами.
- **Исследовательские проекты** – сбор данных из разных источников, проверка гипотез, синтез знаний.
- **Написание кода** – один агент пишет код, другой тестирует, третий ревьюит, а супервайзер решает, когда переходить к следующему этапу.
- **Клиентская поддержка** – супервайзер определяет, нужна ли техническая информация, юридическая консультация или просто разговор с оператором.

В каждом из этих случаев супервайзер не просто передаёт управление, а активно анализирует прогресс и адаптирует стратегию.

### 1.3. Как супервайзер принимает решения

Супервайзер принимает решения на основе **текущего состояния** системы. Состояние включает:

- **Историю сообщений** – что говорили агенты и пользователь.
- **Промежуточные результаты** – факты, гипотезы, ошибки.
- **Метрики прогресса** – сколько информации уже собрано, насколько она полна.

На каждом шаге супервайзер анализирует состояние и выбирает одно из действий:

- **Вызвать агента** – передать задачу конкретному специалисту.
- **Завершить работу** – если цель достигнута.
- **Запросить уточнение у пользователя** – если не хватает данных или есть неоднозначность.

Для этого супервайзер использует языковую модель, которая получает описание текущего состояния и список доступных агентов (как инструментов) и возвращает решение. Это похоже на то, как в Лекции 6.4 мы привязывали инструменты к агенту – теперь агенты сами становятся «инструментами» для супервайзера.

### 1.4. Реализация супервайзера в LangGraph

В LangGraph супервайзер – это обычный узел графа, но с особым поведением. Он:

1. **Принимает состояние** (историю, результаты).
2. **Вызывает LLM** с промптом, который описывает доступных агентов и цель.
3. **Возвращает решение** – имя следующего агента или команду завершить.

На практике супервайзер может быть реализован как:

```python
def supervisor_node(state: dict) -> dict:
    # Получаем историю и текущий прогресс
    messages = state["messages"]
    progress = state.get("progress", "")
    
    # Формируем промпт для супервайзера
    prompt = f"""
    Ты – супервайзер. Управляй командой агентов:
    - researcher: ищет факты
    - critic: проверяет достоверность
    - writer: пишет ответ
    
    История: {messages}
    Прогресс: {progress}
    
    Выбери следующего агента или заверши работу.
    """
    response = llm.invoke(prompt)
    decision = response.content.strip()
    
    # Возвращаем решение
    return {"next_agent": decision}
```

Затем мы добавляем условное ребро, которое направляет управление в зависимости от решения супервайзера. В следующих темах мы детально реализуем этот механизм и покажем, как он делает систему по‑настоящему гибкой.

---

В следующей части мы перейдём к проектированию графа с супервайзером и создадим полноценную оркестрированную multi‑agent систему.

# Лекция 6.7. Супервайзер и сложная оркестрация

## Тема 3. Реализация супервайзера как LLM (скрипт `supervisor_graph_llm.py`)

В предыдущей теме мы спроектировали граф с супервайзером и описали состояние. Теперь мы переходим к реализации супервайзера как **LLM-агента**, который принимает решения на основе анализа текущего состояния и истории. Именно такой подход мы использовали в вашем рабочем скрипте `supervisor_graph_llm.py`, который уже успешно продемонстрировал свою работу.

В этой теме мы детально разберём код, объясним, как супервайзер использует инструменты для вызова агентов, и покажем, как это работает на практике.

---

### 3.1. Почему LLM, а не правила?

Эвристический супервайзер (например, с простыми условиями «если нет фактов → researcher») работает только в самых простых сценариях. В реальных задачах возникает множество нюансов:

- **Неоднозначность** – запрос может требовать и фактов, и анализа, и творчества.
- **Динамика** – промежуточные результаты могут менять план.
- **Контекст** – история диалога может влиять на следующее действие.

LLM-супервайзер преодолевает эти ограничения. Он:

- **Понимает сложные запросы** – например, «Сравни RAG и обычный ChatGPT» может потребовать нескольких шагов исследования, проверки и синтеза.
- **Адаптируется к нестандартным ситуациям** – если критик нашёл противоречия, супервайзер может вернуть исследователя для уточнения.
- **Учитывает историю** – он не повторяет уже выполненные шаги и видит прогресс.

Таким образом, LLM-супервайзер делает систему по‑настоящему гибкой и интеллектуальной, что мы и наблюдаем в вашем работающем коде.

---

### 3.2. Представление агентов как «инструментов»

В LangChain агенты могут вызывать инструменты через `bind_tools()`. В нашем случае супервайзер не выполняет агентов напрямую, а только выбирает, какого агента вызвать. Для этого мы определяем функции-заглушки для каждого агента, которые служат описанием их возможностей. Эти функции привязываются к LLM супервайзера через `bind_tools()`, и модель возвращает `tool_calls` с именем нужного агента.

**В вашем коде это выглядит так:**

```python
from langchain.tools import tool

@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки фактов."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа."""
    return "Писатель напишет ответ."

supervisor_tools = [call_researcher, call_critic, call_writer]
llm_with_tools = llm.bind_tools(supervisor_tools)
```

**Что здесь происходит:**
- Каждая функция снабжена docstring, который описывает её назначение.
- LLM видит эти описания и может решить, какой «инструмент» (агент) нужен.
- `bind_tools()` подготавливает модель к возврату структурированного ответа с `tool_calls`.

Когда супервайзер вызывает LLM, модель возвращает либо `tool_calls` с именем агента, либо текстовый ответ, если она решила завершить работу. Мы парсим `tool_calls` и извлекаем имя агента.

---

### 3.3. Системный промпт для супервайзера

Системный промпт задаёт правила и контекст для LLM. В вашем коде он выглядит так:

```python
SUPERVISOR_PROMPT = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к трём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики

Процесс должен быть логичным:
1. Если нет фактов — вызови researcher.
2. Если факты есть, но нет критики — вызови critic.
3. Если есть и факты, и критика, но нет ответа — вызови writer.
4. Если ответ уже есть — заверши работу.

Твоя задача — анализировать текущее состояние и выбирать правильный следующий шаг.
"""
```

Этот промпт даёт модели чёткое понимание её роли и правил принятия решений. Он также описывает доступные инструменты, что помогает модели выбирать правильный `tool_call`.

---

### 3.4. Полный код супервайзера с LLM (ваш рабочий файл)

Ниже представлен полный код `supervisor_graph_llm.py`, который вы успешно запустили. Он включает все компоненты: состояние, инструменты, узлы, маршрутизацию и тестовый запуск.

```python
"""
supervisor_graph_llm.py - Граф с супервайзером на основе LLM (исправленная версия)
Лекция 6.7, Тема 3
"""

from typing import TypedDict, List, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool

# ============================================================================
# 1. ОПРЕДЕЛЕНИЕ СОСТОЯНИЯ
# ============================================================================

class SupervisorState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    next_agent: Literal["researcher", "critic", "writer", "finish"]
    research_result: Optional[str]
    critic_feedback: Optional[str]
    final_answer: Optional[str]
    iteration: int
    history_summary: str
    error: Optional[str]

# ============================================================================
# 2. ОПРЕДЕЛЕНИЕ «ИНСТРУМЕНТОВ» ДЛЯ СУПЕРВАЙЗЕРА
# ============================================================================

@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов. Используй, если нужна новая информация."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки достоверности и полноты фактов. Вызывай после получения research_result."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа. Вызывай после того, как есть и research_result, и critic_feedback."""
    return "Писатель напишет ответ."

supervisor_tools = [call_researcher, call_critic, call_writer]

# ============================================================================
# 3. ИНИЦИАЛИЗАЦИЯ LLM И ПРОМПТА
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)
llm_with_tools = llm.bind_tools(supervisor_tools)

SUPERVISOR_PROMPT = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к трём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики

Процесс должен быть логичным:
1. Если нет фактов — вызови researcher.
2. Если факты есть, но нет критики — вызови critic.
3. Если есть и факты, и критика, но нет ответа — вызови writer.
4. Если ответ уже есть — заверши работу (не вызывай инструменты).

Твоя задача — анализировать текущее состояние и выбирать правильный следующий шаг.
Ты можешь вызвать только один инструмент за раз (или завершить).
"""

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def supervisor_node(state: SupervisorState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🎯 Супервайзер (итерация {iteration})")

    # Защита от бесконечного цикла (hard limit)
    if iteration > 5:
        print("⚠️  Превышен лимит итераций. Принудительное завершение.")
        return {
            "next_agent": "finish",
            "iteration": iteration,
            "history_summary": f"Итерация {iteration}: завершено принудительно"
        }

    # 🛠 ИСПРАВЛЕНИЕ 1: Немедленный выход, если ответ уже готов
    final_ans = state.get("final_answer")
    if final_ans is not None:
        print("✅ Финальный ответ уже сгенерирован. Завершаем работу.")
        return {
            "next_agent": "finish",
            "iteration": iteration,
            "history_summary": f"Итерация {iteration}: завершено (ответ уже есть)"
        }

    # Формируем контекст для LLM
    question = state.get("question", "")
    research = state.get("research_result")
    critic_fb = state.get("critic_feedback")

    # Строим историю сообщений (для контекста)
    history_text = "\n".join([f"{msg.type}: {msg.content[:200]}" for msg in state.get("messages", [])])

    user_content = f"""
Вопрос пользователя: {question}

Текущий прогресс:
- Исследование: {research if research else "ещё нет"}
- Критика: {critic_fb if critic_fb else "ещё нет"}
- Финальный ответ: {final_ans if final_ans else "ещё нет"}

История сообщений:
{history_text}

Выбери следующего агента (researcher, critic, writer) или заверши работу (finish).
Вызови соответствующий инструмент.
"""

    messages = [
        SystemMessage(content=SUPERVISOR_PROMPT),
        HumanMessage(content=user_content)
    ]

    # Вызов LLM с инструментами
    response = llm_with_tools.invoke(messages)
    print(f"💬 Ответ LLM: {response.content[:100]}...")

    # Извлечение tool_calls
    if hasattr(response, "tool_calls") and response.tool_calls:
        tool_name = response.tool_calls[0]["name"]
        # Преобразуем имя инструмента к нашему формату
        if "researcher" in tool_name:
            proposed = "researcher"
        elif "critic" in tool_name:
            proposed = "critic"
        elif "writer" in tool_name:
            proposed = "writer"
        else:
            proposed = "finish"
    else:
        # Если модель не вызвала инструмент, пытаемся извлечь из текста
        text = response.content.lower()
        if "researcher" in text:
            proposed = "researcher"
        elif "critic" in text:
            proposed = "critic"
        elif "writer" in text:
            proposed = "writer"
        else:
            proposed = "finish"

    # Валидация предложения (корректируем, если логика нарушена)
    if final_ans is not None:
        print("✅ Финальный ответ уже сгенерирован. Завершаем работу (валидация).")
        proposed = "finish"
    elif proposed == "critic" and research is None:
        print("⚠️  LLM вызвала критика без исследования. Перенаправляем на исследователя.")
        proposed = "researcher"
    elif proposed == "writer" and (research is None or critic_fb is None):
        print("⚠️  LLM вызвала писателя без полных данных (исследование или критика отсутствуют). Перенаправляем на критика.")
        if research is None:
            proposed = "researcher"
        else:
            proposed = "critic"
    elif proposed == "researcher" and research is not None and critic_fb is not None:
        print("⚠️  LLM предлагает исследовать, хотя данные уже есть. Перенаправляем на писателя.")
        proposed = "writer"

    # Если после валидации получился "finish", но не все данные собраны — корректируем
    if proposed == "finish" and final_ans is None:
        if research is None:
            proposed = "researcher"
        elif critic_fb is None:
            proposed = "critic"
        else:
            proposed = "writer"

    print(f"🔀 Супервайзер выбрал: {proposed.upper()}")
    return {
        "next_agent": proposed,
        "iteration": iteration,
        "history_summary": f"Итерация {iteration}: выбран {proposed}"
    }


def researcher_node(state: SupervisorState) -> dict:
    print("🔍 Исследователь: собираю факты...")
    question = state.get("question", "")
    # Имитация поиска (можно заменить на реальный RAG)
    if "RAG" in question.upper():
        facts = "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск информации в базе знаний и генерацию текста на основе найденных документов. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
    elif "LLM" in question.upper():
        facts = "Большие языковые модели (LLM) обучаются на больших объёмах текстов и способны генерировать связные ответы, выполнять перевод, реферирование и многие другие задачи."
    else:
        facts = "Информация по запросу не найдена. Возможно, требуется уточнить вопрос."
    msg = AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{facts}")
    return {
        "research_result": facts,
        "messages": [msg]
    }


def critic_node(state: SupervisorState) -> dict:
    print("📊 Критик: проверяю факты...")
    research = state.get("research_result", "")
    if "не найдена" in research.lower():
        feedback = "Факты неполные. Рекомендую провести дополнительный поиск."
    elif len(research) < 30:
        feedback = "Факты слишком краткие. Желательно добавить больше деталей."
    else:
        feedback = "Факты достаточны и достоверны."
    msg = AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")
    return {
        "critic_feedback": feedback,
        "messages": [msg]
    }


def writer_node(state: SupervisorState) -> dict:
    print("✍️  Писатель: пишу финальный ответ...")
    research = state.get("research_result", "")
    feedback = state.get("critic_feedback")
    if feedback is None:
        feedback = "Критика не проводилась."
    else:
        feedback = feedback.lower()

    if "неполные" in feedback or "краткие" in feedback:
        answer = f"На основе найденных фактов:\n{research}\n\nПримечание: {feedback} Рекомендуется уточнить информацию."
    else:
        answer = f"На основе проверенных фактов:\n{research}"
    msg = AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{answer}")
    return {
        "final_answer": answer,
        "messages": [msg]
    }

# ============================================================================
# 5. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_supervisor(state: SupervisorState) -> Literal["researcher", "critic", "writer", "finish"]:
    next_agent = state.get("next_agent", "finish")
    print(f"🔀 Маршрутизация: {next_agent.upper()}")
    return next_agent

# ============================================================================
# 6. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("critic", critic_node)
builder.add_node("writer", writer_node)

builder.set_entry_point("supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "researcher": "researcher",
        "critic": "critic",
        "writer": "writer",
        "finish": END
    }
)
builder.add_edge("researcher", "supervisor")
builder.add_edge("critic", "supervisor")
builder.add_edge("writer", "supervisor")

graph = builder.compile()

# ============================================================================
# 7. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    initial_state = {
        "messages": [HumanMessage(content="Что такое RAG?")],
        "question": "Что такое RAG?",
        "next_agent": "researcher",
        "research_result": None,
        "critic_feedback": None,
        "final_answer": None,
        "iteration": 0,
        "history_summary": "Начало работы",
        "error": None
    }
    config = {"recursion_limit": 10}
    print("=" * 60)
    print("🚀 ЗАПУСК ГРАФА С LLM-СУПЕРВАЙЗЕРОМ (LLM главный)")
    print("=" * 60)
    try:
        result = graph.invoke(initial_state, config=config)
    except Exception as e:
        print(f"❌ Ошибка выполнения: {e}")
        result = None
    if result:
        print("\n" + "=" * 60)
        print("✅ ИТОГОВЫЙ РЕЗУЛЬТАТ")
        print("=" * 60)
        print(f"Финальный ответ: {result.get('final_answer', 'Не сгенерирован')}")
        print(f"Количество шагов: {len(result['messages'])}")
        print("\n📋 Полная история сообщений:")
        for i, msg in enumerate(result["messages"], 1):
            print(f"{i}. {msg.content[:100]}...")
```

---

### 3.5. Как это работает

1. **Супервайзер** (узел `supervisor_node`) получает состояние, включая вопрос, результаты агентов и историю.
2. Он формирует промпт с текущим прогрессом и вызывает LLM с привязанными инструментами.
3. LLM возвращает `tool_calls`, указывая, какого агента вызвать, или текстовый ответ, если решает завершить.
4. Мы извлекаем имя агента из `tool_calls` и сохраняем в `next_agent`.
5. **Валидация** корректирует ошибочные решения (например, если LLM вызывает критика без исследования).
6. **Маршрутизация** направляет управление соответствующему агенту.
7. После выполнения агента управление возвращается к супервайзеру.
8. Цикл повторяется, пока супервайзер не решит завершить работу (или не сработает защита от цикла).

**Результат работы вашего скрипта:**

```
🚀 ЗАПУСК ГРАФА С LLM-СУПЕРВАЙЗЕРОМ (LLM главный)
============================================================

🎯 Супервайзер (итерация 1)
💬 Ответ LLM: ...
🔀 Супервайзер выбрал: RESEARCHER
🔀 Маршрутизация: RESEARCHER
🔍 Исследователь: собираю факты...

🎯 Супервайзер (итерация 2)
💬 Ответ LLM: ...
🔀 Супервайзер выбрал: CRITIC
🔀 Маршрутизация: CRITIC
📊 Критик: проверяю факты...

🎯 Супервайзер (итерация 3)
💬 Ответ LLM: ...
🔀 Супервайзер выбрал: WRITER
🔀 Маршрутизация: WRITER
✍️  Писатель: пишу финальный ответ...

🎯 Супервайзер (итерация 4)
✅ Финальный ответ уже сгенерирован. Завершаем работу.
🔀 Маршрутизация: FINISH

============================================================
✅ ИТОГОВЫЙ РЕЗУЛЬТАТ
============================================================
Финальный ответ: На основе проверенных фактов:
RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск информации в базе знаний и генерацию текста на основе найденных документов. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM).
```

---

### 3.6. Преимущества LLM-супервайзера

| Аспект | Эвристический супервайзер | LLM-супервайзер |
|--------|---------------------------|-----------------|
| **Гибкость** | Ограничена заданными правилами | Адаптируется к контексту |
| **Понимание** | Не понимает нюансов | Учитывает смысл и историю |
| **Масштабируемость** | Трудно добавить новые сценарии | Легко добавить новых агентов через инструменты |
| **Качество решений** | Предсказуемо, но не всегда оптимально | Может принимать нестандартные решения |

Ваш код демонстрирует, как LLM-супервайзер успешно управляет процессом, последовательно вызывая нужных агентов и завершая работу, когда ответ готов.

---

## Краткий итог Тема 3

- **LLM-супервайзер** заменяет эвристические правила, делая управление более интеллектуальным.
- **Агенты представлены как инструменты** через `bind_tools()`, что позволяет супервайзеру «вызывать» их.
- **Валидация решений** предотвращает логические ошибки (например, вызов критика без исследования).
- **Граф** сохраняет структуру: супервайзер → агент → супервайзер → … → завершение.
- **Защита от бесконечного цикла** реализована через `iteration` и `max_iterations`.

Теперь ваша система стала по‑настоящему адаптивной. В следующей теме мы рассмотрим, как супервайзер может управлять более сложными сценариями, включая параллельные вызовы агентов.


## Тема 4. Динамическое добавление агентов и маршрутизация (скрипт `supervisor_graph_dynamic.py`)

В предыдущей теме мы реализовали супервайзера, который управлял тремя фиксированными агентами: исследователем, критиком и писателем. Это была хорошая основа, но в реальных системах состав агентов может меняться в зависимости от задачи. Иногда нам нужен калькулятор, иногда — эксперт по данным, а иногда — редактор текста.

В этой теме мы покажем, как супервайзер может **динамически выбирать агентов** из расширяемого списка, адаптируясь к контексту запроса. Для этого мы добавим нового агента – **калькулятор** – и продемонстрируем, как супервайзер сам решает, когда вызывать исследователя, а когда калькулятора.

Это особенно полезно в сценариях, где:
- **Задачи требуют разных навыков** – например, для математического вопроса нужен калькулятор, а для исторического – исследователь.
- **Система должна легко расширяться** – добавление нового агента не должно требовать изменения кода супервайзера.
- **Агенты могут подключаться динамически** – в зависимости от доступных ресурсов или настроек пользователя.

---

### 4.1. Расширение состояния и добавление агента

Чтобы добавить нового агента, мы:

1. **Расширяем состояние** – добавляем поле `calculation_result` для хранения результатов вычислений и поле `last_agent` для защиты от повторных вызовов одного агента.
2. **Определяем новый инструмент** для супервайзера с описанием возможностей калькулятора.
3. **Создаём узел-агент** `calculator_node`, который выполняет вычисления.
4. **Обновляем маршрутизацию** – добавляем `"calculator"` в перечень возможных значений `next_agent`.

```python
class SupervisorState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    next_agent: Literal["researcher", "critic", "writer", "calculator", "finish"]
    research_result: Optional[str]
    critic_feedback: Optional[str]
    final_answer: Optional[str]
    calculation_result: Optional[str]
    iteration: int
    history_summary: str
    error: Optional[str]
    last_agent: Optional[str]   # предотвращает повторный вызов одного агента
```

Поле `last_agent` позволяет супервайзеру избегать зацикливания – если он предлагает того же агента, что и на предыдущем шаге, система автоматически переключается на другого.

---

### 4.2. Инструменты для супервайзера

Супервайзер видит четырёх агентов как инструменты с чёткими описаниями. Это позволяет LLM выбирать подходящего агента в зависимости от вопроса:

```python
@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов. Используй, если нужна информация из документов или интернета."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки достоверности и полноты фактов. Вызывай после получения research_result."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа. Вызывай после того, как есть и research_result, и critic_feedback."""
    return "Писатель напишет ответ."

@tool
def call_calculator(expression: str) -> str:
    """
    Вызывает агента-калькулятора для выполнения математических вычислений.
    Используй, когда вопрос требует вычислений: проценты, корни, тригонометрия и т.д.
    """
    return "Калькулятор выполнит вычисления."

supervisor_tools = [call_researcher, call_critic, call_writer, call_calculator]
llm_with_tools = llm.bind_tools(supervisor_tools)
```

Теперь супервайзер может выбирать между четырьмя агентами. Его системный промпт описывает правила выбора в зависимости от типа вопроса:

```python
SUPERVISOR_PROMPT = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к четырём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу (исторические, технические, фактологические)
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики
- call_calculator: выполняет математические вычисления (проценты, корни, суммы)

Правила принятия решений:
1. Если вопрос явно требует вычислений (проценты, арифметика) — сначала вызови calculator.
2. Если вопрос требует фактических знаний — вызови researcher.
3. Если есть research_result, но нет critic_feedback — вызови critic.
4. Если есть research_result и critic_feedback, но нет final_answer — вызови writer.
5. Если есть calculation_result, но нет final_answer — вызови writer (критика не нужна).
6. Если final_answer уже есть — заверши работу (finish).
7. Не вызывай одного агента дважды подряд, если только это не необходимо.

Ты можешь вызвать только один инструмент за раз (или завершить).
"""
```

---

### 4.3. Детерминированная логика принятия решений

В нашем подходе супервайзер **не полагается исключительно на LLM**. Вместо этого в `supervisor_node` сначала применяются **детерминированные правила**, которые гарантируют логичный порядок шагов и предотвращают зацикливание:

```python
def supervisor_node(state: SupervisorState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🎯 Супервайзер (итерация {iteration})")

    # Защита от бесконечного цикла
    if iteration > 8:
        print("⚠️  Превышен лимит итераций. Принудительное завершение.")
        return {"next_agent": "finish", "iteration": iteration}

    final_ans = state.get("final_answer")
    if final_ans is not None:
        print("✅ Финальный ответ уже сгенерирован. Завершаем работу.")
        return {"next_agent": "finish", "iteration": iteration}

    research = state.get("research_result")
    critic_fb = state.get("critic_feedback")
    calc_result = state.get("calculation_result")
    last_agent = state.get("last_agent")

    # ДЕТЕРМИНИРОВАННАЯ ЛОГИКА (приоритет выше LLM)

    # Если есть вычисления, но нет ответа → writer
    if calc_result is not None and final_ans is None:
        print("🧮 Есть результат вычислений → направляем к писателю.")
        proposed = "writer"

    # Если есть исследование, но нет критики → critic
    elif research is not None and critic_fb is None and final_ans is None:
        print("📚 Есть исследование, нет критики → направляем к критику.")
        proposed = "critic"

    # Если есть исследование и критика, но нет ответа → writer
    elif research is not None and critic_fb is not None and final_ans is None:
        print("📚✅ Есть исследование и критика → направляем к писателю.")
        proposed = "writer"

    # Если нет ничего → решаем через LLM, что первым вызвать
    else:
        # LLM выбирает первого агента
        user_content = f"Вопрос: {state['question']}\nПрогресс: исследование {research if research else 'нет'}, критика {critic_fb if critic_fb else 'нет'}, вычисления {calc_result if calc_result else 'нет'}"
        messages = [SystemMessage(content=SUPERVISOR_PROMPT), HumanMessage(content=user_content)]
        response = llm_with_tools.invoke(messages)
        if hasattr(response, "tool_calls") and response.tool_calls:
            tool_name = response.tool_calls[0]["name"]
            if "researcher" in tool_name:
                proposed = "researcher"
            elif "calculator" in tool_name:
                proposed = "calculator"
            else:
                proposed = "researcher"
        else:
            proposed = "researcher" if "calculator" not in response.content.lower() else "calculator"

    # Защита от повторов
    if proposed == last_agent and proposed not in ("finish", "writer"):
        proposed = "calculator" if proposed == "researcher" else "researcher"

    # Если всё готово, но почему-то не writer → принудительно writer
    if (research is not None and critic_fb is not None and final_ans is None) or (calc_result is not None and final_ans is None):
        if proposed not in ("writer", "finish"):
            proposed = "writer"

    print(f"🔀 Супервайзер выбрал: {proposed.upper()}")
    return {
        "next_agent": proposed,
        "iteration": iteration,
        "history_summary": f"Итерация {iteration}: выбран {proposed}",
        "last_agent": proposed
    }
```

Этот подход даёт несколько преимуществ:
- **Предсказуемость** – после сбора фактов всегда следует проверка, а затем написание ответа.
- **Защита от ошибок LLM** – даже если модель предложит нелогичный шаг, детерминированные правила скорректируют его.
- **Эффективность** – меньше итераций, так как не нужно каждый раз спрашивать модель.

---

### 4.4. Узлы-агенты

Все агенты реализованы как отдельные функции-узлы. Рассмотрим нового агента – калькулятор:

```python
def calculator_node(state: SupervisorState) -> dict:
    print("🧮 Калькулятор: выполняю вычисления...")
    question = state.get("question", "")
    
    # Простой парсер математических выражений
    try:
        expr = question.lower()
        result = None
        
        # Обработка процентов
        if '%' in expr and 'от' in expr:
            parts = expr.split('от')
            if len(parts) == 2:
                percent = re.sub(r'[^0-9.]', '', parts[0].strip())
                number = re.sub(r'[^0-9.]', '', parts[1].strip())
                if percent and number:
                    result = float(number) * float(percent) / 100
        else:
            numbers = re.findall(r'[\d.]+', expr)
            if len(numbers) >= 2:
                if 'плюс' in expr or '+' in expr:
                    result = sum(float(n) for n in numbers)
                elif 'минус' in expr or '-' in expr:
                    result = float(numbers[0]) - float(numbers[1])
                elif 'умножить' in expr or '*' in expr:
                    result = float(numbers[0]) * float(numbers[1])
                elif 'делить' in expr or '/' in expr:
                    result = float(numbers[0]) / float(numbers[1]) if float(numbers[1]) != 0 else "Ошибка: деление на ноль"
        
        if result is not None:
            calc_result = f"Результат вычисления: {result}"
        else:
            calc_result = "Не удалось распознать математическое выражение. Уточните вопрос."
    except Exception as e:
        calc_result = f"Ошибка вычисления: {e}"

    return {
        "calculation_result": calc_result,
        "messages": [AIMessage(content=f"🧮 РЕЗУЛЬТАТ ВЫЧИСЛЕНИЯ:\n{calc_result}")]
    }
```

Остальные агенты (`researcher`, `critic`, `writer`) работают аналогично предыдущим темам.

---

### 4.5. Полный код `supervisor_graph_dynamic.py`

Ниже представлен полный код с динамическим выбором агентов. Он включает супервайзера, исследователя, критика, писателя и калькулятора, а также тестовый запуск на трёх типах вопросов.

```python
"""
supervisor_graph_dynamic.py - Граф с динамическим выбором агентов
Лекция 6.7, Тема 4

Добавлен новый агент-калькулятор. Супервайзер сам решает, какого агента вызвать.
Использованы детерминированные правила для предотвращения зацикливания.
"""

from typing import TypedDict, List, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
import math
import re

# ============================================================================
# 1. ОПРЕДЕЛЕНИЕ СОСТОЯНИЯ
# ============================================================================

class SupervisorState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    next_agent: Literal["researcher", "critic", "writer", "calculator", "finish"]
    research_result: Optional[str]
    critic_feedback: Optional[str]
    final_answer: Optional[str]
    calculation_result: Optional[str]
    iteration: int
    history_summary: str
    error: Optional[str]
    last_agent: Optional[str]

# ============================================================================
# 2. ИНСТРУМЕНТЫ ДЛЯ СУПЕРВАЙЗЕРА
# ============================================================================

@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов. Используй, если нужна информация из документов или интернета."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки достоверности и полноты фактов. Вызывай после получения research_result."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа. Вызывай после того, как есть и research_result, и critic_feedback."""
    return "Писатель напишет ответ."

@tool
def call_calculator(expression: str) -> str:
    """
    Вызывает агента-калькулятора для выполнения математических вычислений.
    Используй, когда вопрос требует вычислений: проценты, корни, тригонометрия и т.д.
    """
    return "Калькулятор выполнит вычисления."

supervisor_tools = [call_researcher, call_critic, call_writer, call_calculator]

# ============================================================================
# 3. ИНИЦИАЛИЗАЦИЯ LLM И ПРОМПТА
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)
llm_with_tools = llm.bind_tools(supervisor_tools)

SUPERVISOR_PROMPT = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к четырём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу (исторические, технические, фактологические)
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики
- call_calculator: выполняет математические вычисления (проценты, корни, суммы)

Правила принятия решений:
1. Если вопрос явно требует вычислений (проценты, арифметика) — сначала вызови calculator.
2. Если вопрос требует фактических знаний — вызови researcher.
3. Если есть research_result, но нет critic_feedback — вызови critic.
4. Если есть research_result и critic_feedback, но нет final_answer — вызови writer.
5. Если есть calculation_result, но нет final_answer — вызови writer (критика не нужна).
6. Если final_answer уже есть — заверши работу (finish).
7. Не вызывай одного агента дважды подряд, если только это не необходимо.

Ты можешь вызвать только один инструмент за раз (или завершить).
"""

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def supervisor_node(state: SupervisorState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🎯 Супервайзер (итерация {iteration})")

    # Защита от бесконечного цикла
    if iteration > 8:
        print("⚠️  Превышен лимит итераций. Принудительное завершение.")
        return {
            "next_agent": "finish",
            "iteration": iteration,
            "history_summary": f"Итерация {iteration}: завершено принудительно",
            "last_agent": state.get("last_agent")
        }

    final_ans = state.get("final_answer")
    if final_ans is not None:
        print("✅ Финальный ответ уже сгенерирован. Завершаем работу.")
        return {
            "next_agent": "finish",
            "iteration": iteration,
            "history_summary": f"Итерация {iteration}: завершено (ответ уже есть)",
            "last_agent": state.get("last_agent")
        }

    question = state.get("question", "")
    research = state.get("research_result")
    critic_fb = state.get("critic_feedback")
    calc_result = state.get("calculation_result")
    last_agent = state.get("last_agent")

    # ----------------------------------------------------------------------
    # ДЕТЕРМИНИРОВАННАЯ ЛОГИКА (приоритет выше LLM)
    # ----------------------------------------------------------------------

    # Если есть вычисления, но нет ответа → writer
    if calc_result is not None and final_ans is None:
        print("🧮 Есть результат вычислений → направляем к писателю.")
        proposed = "writer"

    # Если есть исследование, но нет критики → critic
    elif research is not None and critic_fb is None and final_ans is None:
        print("📚 Есть исследование, нет критики → направляем к критику.")
        proposed = "critic"

    # Если есть исследование и критика, но нет ответа → writer
    elif research is not None and critic_fb is not None and final_ans is None:
        print("📚✅ Есть исследование и критика → направляем к писателю.")
        proposed = "writer"

    # Если нет ничего → решаем через LLM, что первым вызвать
    else:
        # Формируем контекст для LLM
        history_text = "\n".join([f"{msg.type}: {msg.content[:200]}" for msg in state.get("messages", [])])

        user_content = f"""
Вопрос пользователя: {question}

Текущий прогресс:
- Исследование: {research if research else "ещё нет"}
- Критика: {critic_fb if critic_fb else "ещё нет"}
- Вычисления: {calc_result if calc_result else "ещё нет"}
- Финальный ответ: {final_ans if final_ans else "ещё нет"}

История сообщений:
{history_text}

Выбери первого агента для обработки вопроса (researcher или calculator).
Не вызывай critic или writer без соответствующих данных.
"""

        messages = [
            SystemMessage(content=SUPERVISOR_PROMPT),
            HumanMessage(content=user_content)
        ]

        response = llm_with_tools.invoke(messages)
        print(f"💬 Ответ LLM: {response.content[:100]}...")

        # Извлечение предложения от LLM
        if hasattr(response, "tool_calls") and response.tool_calls:
            tool_name = response.tool_calls[0]["name"]
            if "researcher" in tool_name:
                proposed = "researcher"
            elif "calculator" in tool_name:
                proposed = "calculator"
            else:
                # Если LLM выбрала что-то другое, перенаправляем на researcher
                proposed = "researcher"
        else:
            text = response.content.lower()
            if "calculator" in text or "calculate" in text:
                proposed = "calculator"
            else:
                proposed = "researcher"

    # ----------------------------------------------------------------------
    # ДОПОЛНИТЕЛЬНАЯ ЗАЩИТА ОТ ПОВТОРОВ
    # ----------------------------------------------------------------------
    if proposed == last_agent and proposed not in ("finish", "writer"):
        # Если предлагается тот же агент, что и на прошлом шаге, переключаем
        print(f"⚠️  Предложен повторный вызов {proposed}, перенаправляем.")
        if proposed == "researcher":
            proposed = "calculator" if calc_result is None else "writer"
        elif proposed == "calculator":
            proposed = "researcher" if research is None else "writer"
        elif proposed == "critic":
            proposed = "writer"

    # ----------------------------------------------------------------------
    # ФИНАЛЬНАЯ ПРОВЕРКА
    # ----------------------------------------------------------------------
    # Если всё готово, но почему-то не writer → принудительно writer
    if (research is not None and critic_fb is not None and final_ans is None) or \
       (calc_result is not None and final_ans is None):
        if proposed not in ("writer", "finish"):
            print("🔄 Принудительное перенаправление к писателю (все данные готовы).")
            proposed = "writer"

    print(f"🔀 Супервайзер выбрал: {proposed.upper()}")
    return {
        "next_agent": proposed,
        "iteration": iteration,
        "history_summary": f"Итерация {iteration}: выбран {proposed}",
        "last_agent": proposed
    }


def researcher_node(state: SupervisorState) -> dict:
    print("🔍 Исследователь: собираю факты...")
    question = state.get("question", "")
    if "RAG" in question.upper():
        facts = "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск информации в базе знаний и генерацию текста. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
    elif "LLM" in question.upper():
        facts = "Большие языковые модели (LLM) обучаются на больших объёмах текстов и способны генерировать связные ответы."
    else:
        facts = "Информация по запросу не найдена. Возможно, требуется уточнить вопрос."
    return {
        "research_result": facts,
        "messages": [AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{facts}")]
    }


def critic_node(state: SupervisorState) -> dict:
    print("📊 Критик: проверяю факты...")
    research = state.get("research_result", "")
    if "не найдена" in research.lower():
        feedback = "Факты неполные. Рекомендую дополнительный поиск."
    elif len(research) < 30:
        feedback = "Факты слишком краткие. Добавьте детали."
    else:
        feedback = "Факты достаточны и достоверны."
    return {
        "critic_feedback": feedback,
        "messages": [AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")]
    }


def calculator_node(state: SupervisorState) -> dict:
    print("🧮 Калькулятор: выполняю вычисления...")
    question = state.get("question", "")
    
    # Простой парсер
    try:
        expr = question.lower()
        result = None
        
        # Обработка процентов
        if '%' in expr and 'от' in expr:
            parts = expr.split('от')
            if len(parts) == 2:
                percent = re.sub(r'[^0-9.]', '', parts[0].strip())
                number = re.sub(r'[^0-9.]', '', parts[1].strip())
                if percent and number:
                    result = float(number) * float(percent) / 100
        else:
            numbers = re.findall(r'[\d.]+', expr)
            if len(numbers) >= 2:
                if 'плюс' in expr or '+' in expr:
                    result = sum(float(n) for n in numbers)
                elif 'минус' in expr or '-' in expr:
                    result = float(numbers[0]) - float(numbers[1])
                elif 'умножить' in expr or '*' in expr:
                    result = float(numbers[0]) * float(numbers[1])
                elif 'делить' in expr or '/' in expr:
                    result = float(numbers[0]) / float(numbers[1]) if float(numbers[1]) != 0 else "Ошибка: деление на ноль"
        
        if result is not None:
            calc_result = f"Результат вычисления: {result}"
        else:
            calc_result = "Не удалось распознать математическое выражение. Уточните вопрос."
    except Exception as e:
        calc_result = f"Ошибка вычисления: {e}"

    return {
        "calculation_result": calc_result,
        "messages": [AIMessage(content=f"🧮 РЕЗУЛЬТАТ ВЫЧИСЛЕНИЯ:\n{calc_result}")]
    }


def writer_node(state: SupervisorState) -> dict:
    print("✍️  Писатель: пишу финальный ответ...")
    research = state.get("research_result", "")
    feedback = state.get("critic_feedback")
    calc = state.get("calculation_result", "")
    
    parts = []
    if research:
        parts.append(f"Факты:\n{research}")
    if calc:
        parts.append(f"Вычисления:\n{calc}")
    if feedback:
        parts.append(f"Примечания: {feedback}")
    
    answer = "\n\n".join(parts) if parts else "Недостаточно данных для ответа."
    
    return {
        "final_answer": answer,
        "messages": [AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{answer}")]
    }

# ============================================================================
# 5. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_supervisor(state: SupervisorState) -> Literal["researcher", "critic", "writer", "calculator", "finish"]:
    next_agent = state.get("next_agent", "finish")
    print(f"🔀 Маршрутизация: {next_agent.upper()}")
    return next_agent

# ============================================================================
# 6. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("critic", critic_node)
builder.add_node("writer", writer_node)
builder.add_node("calculator", calculator_node)

builder.set_entry_point("supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "researcher": "researcher",
        "critic": "critic",
        "writer": "writer",
        "calculator": "calculator",
        "finish": END
    }
)
builder.add_edge("researcher", "supervisor")
builder.add_edge("critic", "supervisor")
builder.add_edge("writer", "supervisor")
builder.add_edge("calculator", "supervisor")

graph = builder.compile()

# ============================================================================
# 7. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    test_questions = [
        "Что такое RAG?",
        "Сколько будет 25% от 200?",
        "Какая погода в Москве?"
    ]
    
    for q in test_questions:
        initial_state = {
            "messages": [HumanMessage(content=q)],
            "question": q,
            "next_agent": "researcher",
            "research_result": None,
            "critic_feedback": None,
            "final_answer": None,
            "calculation_result": None,
            "iteration": 0,
            "history_summary": "Начало работы",
            "error": None,
            "last_agent": None
        }
        config = {"recursion_limit": 15}
        
        print("\n" + "=" * 60)
        print(f"📝 Вопрос: {q}")
        print("=" * 60)
        
        try:
            result = graph.invoke(initial_state, config=config)
            print(f"\n✅ Финальный ответ:\n{result.get('final_answer', 'Не сгенерирован')}")
            print(f"Шагов: {result.get('iteration', 0)}")
        except Exception as e:
            print(f"❌ Ошибка: {e}")
```

---

### 4.6. Результаты тестирования

При запуске на трёх вопросах система показала:

| Вопрос | Выбранный агент | Результат |
|--------|-----------------|-----------|
| «Что такое RAG?» | `researcher` → `critic` → `writer` | Фактический ответ с проверкой |
| «Сколько будет 25% от 200?» | `calculator` → `writer` | Математический расчёт |
| «Какая погода в Москве?» | `researcher` → `critic` → `writer` | Ответ с пометкой о неполноте данных |

**Вывод:** супервайзер успешно определяет, когда нужен исследователь, а когда калькулятор, и строит соответствующий маршрут. При этом он всегда завершает работу писателем, который формирует финальный ответ.

---

## Краткий итог Тема 4

- **Супервайзер может динамически выбирать агентов** – для этого мы добавили калькулятор и описали его в инструментах.
- **Детерминированные правила** обеспечивают предсказуемый порядок шагов и защиту от ошибок LLM.
- **Состояние расширено** полями `calculation_result` и `last_agent` для хранения результатов вычислений и предотвращения повторных вызовов.
- **Тестирование подтвердило** – система корректно обрабатывает фактологические, математические и неопределённые запросы.

Теперь вы можете легко добавлять новых агентов, просто описывая их как инструменты и создавая соответствующие узлы. Это делает архитектуру гибкой и масштабируемой.

---

**В следующей теме мы рассмотрим, как супервайзер может управлять параллельными вызовами агентов и комбинировать их результаты.**


## Тема 5. Мониторинг, логирование и человеческий контроль (скрипт `supervised_multi_agent.py`)

Мы построили систему, где супервайзер динамически управляет командой агентов. Теперь настало время сделать её прозрачной, управляемой и готовой к реальной эксплуатации. В этой финальной теме мы добавим три ключевых элемента:

1. **Мониторинг и логирование** – будем записывать каждое решение супервайзера, результаты агентов и время выполнения, чтобы анализировать работу системы.
2. **Человеческий контроль (Human‑in‑the‑loop)** – оператор сможет подтверждать или отклонять решения супервайзера перед выполнением, что критично для ответственных задач.
3. **Итоговый класс `SupervisedMultiAgent`** – объединим всё в единый, готовый к использованию компонент с поддержкой памяти и конфигурации.

---

### 5.1. Логирование решений и результатов

Для логирования мы используем стандартный модуль `logging` и кастомный логгер, который записывает события в файл и выводит в консоль. Будем логировать:

- Каждый вызов супервайзера (итерация, выбранный агент).
- Начало и завершение работы каждого агента (с результатом).
- Время выполнения каждого шага.
- Ошибки и предупреждения.

В коде мы встроим вызовы логгера в узлы графа.

---

### 5.2. Человеческий контроль (Human‑in‑the‑loop)

В LangGraph есть встроенный механизм прерываний через `interrupt_before` или `interrupt_after` при компиляции графа. Мы можем остановить выполнение **перед узлом супервайзера**, чтобы оператор мог увидеть предложенное действие и либо подтвердить, либо изменить его.

Для этого мы:

1. Компилируем граф с `checkpointer` (например, `MemorySaver`).
2. Указываем `interrupt_before=["supervisor"]`.
3. При запуске графа он остановится перед выполнением `supervisor_node`. Мы можем получить текущее состояние, посмотреть, кого супервайзер хочет вызвать, и запросить подтверждение у пользователя.
4. После подтверждения продолжаем выполнение, вызвав `graph.invoke(None, config=config)`.

Это позволяет реализовать безопасный контроль над действиями агента.

---

### 5.3. Интеграция с LangSmith

LangSmith – это платформа для трассировки LLM-приложений. Мы уже использовали её в Лекции 6.5. В нашей системе с супервайзером она автоматически запишет все вызовы LLM, если настроены переменные окружения. Мы просто упомянем это и покажем, как включить.

---

### 5.4. Итоговый класс `SupervisedMultiAgent`

Теперь соберём всё в единый класс, который инкапсулирует:

- Инициализацию LLM, инструментов, состояния и графа.
- Настройку логирования.
- Возможность включения Human‑in‑the‑loop.
- Методы для синхронного и асинхронного запуска.
- Сохранение состояния между сессиями через `MemorySaver` (или `SqliteSaver`).

Класс будет принимать параметры: `model_name`, `temperature`, `use_human_loop`, `log_level` и т.д.

---

## Полный код `supervised_multi_agent.py`

Ниже представлен полный код класса `SupervisedMultiAgent` с комментариями. Он объединяет все темы лекции: супервайзера, агентов, динамическую маршрутизацию, логирование и человеческий контроль.

```python
"""
supervised_multi_agent.py - Итоговый класс для системы с супервайзером
Лекция 6.7, Тема 5

Включает:
- Супервайзера с динамическим выбором агентов
- Агентов: исследователь, критик, писатель, калькулятор
- Логирование каждого шага
- Human‑in‑the‑loop (подтверждение действий супервайзера)
- Память через MemorySaver
- Готов к использованию в продакшене
"""

import logging
import time
import re
import math
from typing import TypedDict, List, Annotated, Literal, Optional, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

# ============================================================================
# 1. КОНФИГУРАЦИЯ ЛОГГЕРА
# ============================================================================

def setup_logger(log_file: str = "supervisor.log", log_level: int = logging.INFO):
    """Настраивает логирование в файл и консоль."""
    logger = logging.getLogger("SupervisedMultiAgent")
    logger.setLevel(log_level)
    logger.handlers.clear()
    
    # Хэндлер для файла
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setLevel(log_level)
    fh.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
    logger.addHandler(fh)
    
    # Хэндлер для консоли
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
    logger.addHandler(ch)
    
    return logger

# ============================================================================
# 2. ОПРЕДЕЛЕНИЕ СОСТОЯНИЯ
# ============================================================================

class SupervisorState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    next_agent: Literal["researcher", "critic", "writer", "calculator", "finish"]
    research_result: Optional[str]
    critic_feedback: Optional[str]
    final_answer: Optional[str]
    calculation_result: Optional[str]
    iteration: int
    history_summary: str
    error: Optional[str]
    last_agent: Optional[str]

# ============================================================================
# 3. ОПРЕДЕЛЕНИЕ ИНСТРУМЕНТОВ ДЛЯ СУПЕРВАЙЗЕРА
# ============================================================================

@tool
def call_researcher(question: str) -> str:
    """Вызывает агента-исследователя для сбора фактов. Используй, если нужна информация из документов или интернета."""
    return "Исследователь соберёт факты по запросу."

@tool
def call_critic(facts: str) -> str:
    """Вызывает агента-критика для проверки достоверности и полноты фактов. Вызывай после получения research_result."""
    return "Критик проверит факты."

@tool
def call_writer(facts: str, feedback: str) -> str:
    """Вызывает агента-писателя для написания финального ответа. Вызывай после того, как есть и research_result, и critic_feedback."""
    return "Писатель напишет ответ."

@tool
def call_calculator(expression: str) -> str:
    """
    Вызывает агента-калькулятора для выполнения математических вычислений.
    Используй, когда вопрос требует вычислений: проценты, корни, тригонометрия и т.д.
    """
    return "Калькулятор выполнит вычисления."

# ============================================================================
# 4. КЛАСС SUPERVISEDMULTIAGENT
# ============================================================================

class SupervisedMultiAgent:
    """
    Полноценная multi‑agent система с супервайзером, логированием,
    человеческим контролем и памятью.
    """
    
    def __init__(
        self,
        model_name: str = "qwen2.5:3b",
        temperature: float = 0.0,
        max_iterations: int = 8,
        use_human_loop: bool = False,
        log_file: str = "supervisor.log",
        log_level: int = logging.INFO,
        use_sqlite: bool = False
    ):
        """
        Инициализация системы.
        
        Аргументы:
            model_name: имя модели в Ollama
            temperature: температура генерации
            max_iterations: максимальное число итераций
            use_human_loop: включить ли подтверждение действий супервайзера
            log_file: путь к файлу логов
            log_level: уровень логирования
            use_sqlite: использовать SqliteSaver вместо MemorySaver
        """
        self.max_iterations = max_iterations
        self.use_human_loop = use_human_loop
        
        # Настройка логгера
        self.logger = setup_logger(log_file, log_level)
        self.logger.info("🚀 Инициализация SupervisedMultiAgent")
        
        # Инициализация LLM
        self.llm = ChatOllama(model=model_name, temperature=temperature, num_predict=512)
        self.llm_with_tools = self.llm.bind_tools([
            call_researcher, call_critic, call_writer, call_calculator
        ])
        
        # Системный промпт для супервайзера
        self.supervisor_prompt = """
Ты — супервайзер, который управляет командой агентов для ответа на вопрос пользователя.

У тебя есть доступ к четырём агентам (вызываются через инструменты):
- call_researcher: собирает факты по вопросу (исторические, технические, фактологические)
- call_critic: проверяет факты на достоверность и полноту
- call_writer: пишет финальный ответ на основе фактов и критики
- call_calculator: выполняет математические вычисления (проценты, корни, суммы)

Правила принятия решений:
1. Если вопрос явно требует вычислений (проценты, арифметика) — сначала вызови calculator.
2. Если вопрос требует фактических знаний — вызови researcher.
3. Если есть research_result, но нет critic_feedback — вызови critic.
4. Если есть research_result и critic_feedback, но нет final_answer — вызови writer.
5. Если есть calculation_result, но нет final_answer — вызови writer (критика не нужна).
6. Если final_answer уже есть — заверши работу (finish).
7. Не вызывай одного агента дважды подряд, если только это не необходимо.

Ты можешь вызвать только один инструмент за раз (или завершить).
"""
        
        # Строим граф
        self.graph = self._build_graph()
        self.logger.info("✅ Граф готов")
    
    def _build_graph(self):
        """Строит граф с супервайзером и агентами, добавляет MemorySaver."""
        
        # Определяем узлы
        def supervisor_node(state: SupervisorState) -> dict:
            iteration = state.get("iteration", 0) + 1
            self.logger.info(f"🔄 Итерация {iteration}")
            
            if iteration > self.max_iterations:
                self.logger.warning("Превышен лимит итераций, принудительное завершение")
                return {"next_agent": "finish", "iteration": iteration, "last_agent": state.get("last_agent")}
            
            final_ans = state.get("final_answer")
            if final_ans is not None:
                self.logger.info("✅ Финальный ответ уже есть, завершаем")
                return {"next_agent": "finish", "iteration": iteration, "last_agent": state.get("last_agent")}
            
            research = state.get("research_result")
            critic_fb = state.get("critic_feedback")
            calc_result = state.get("calculation_result")
            last_agent = state.get("last_agent")
            
            # Детерминированная логика
            if calc_result is not None and final_ans is None:
                proposed = "writer"
                self.logger.info("🧮 Есть вычисления → писатель")
            elif research is not None and critic_fb is None and final_ans is None:
                proposed = "critic"
                self.logger.info("📚 Есть исследование, нет критики → критик")
            elif research is not None and critic_fb is not None and final_ans is None:
                proposed = "writer"
                self.logger.info("📚✅ Есть исследование и критика → писатель")
            else:
                # LLM выбирает первого агента
                user_content = f"Вопрос: {state['question']}\nПрогресс: исследование {research if research else 'нет'}, критика {critic_fb if critic_fb else 'нет'}, вычисления {calc_result if calc_result else 'нет'}"
                messages = [SystemMessage(content=self.supervisor_prompt), HumanMessage(content=user_content)]
                response = self.llm_with_tools.invoke(messages)
                if hasattr(response, "tool_calls") and response.tool_calls:
                    tool_name = response.tool_calls[0]["name"]
                    if "researcher" in tool_name:
                        proposed = "researcher"
                    elif "calculator" in tool_name:
                        proposed = "calculator"
                    else:
                        proposed = "researcher"
                else:
                    proposed = "researcher" if "calculator" not in response.content.lower() else "calculator"
                self.logger.info(f"💬 LLM выбрала: {proposed}")
            
            # Защита от повторов
            if proposed == last_agent and proposed not in ("finish", "writer"):
                self.logger.warning(f"⚠️ Повторный вызов {proposed}, переключаем")
                if proposed == "researcher":
                    proposed = "calculator" if calc_result is None else "writer"
                elif proposed == "calculator":
                    proposed = "researcher" if research is None else "writer"
                elif proposed == "critic":
                    proposed = "writer"
            
            # Принудительный writer, если всё готово
            if (research is not None and critic_fb is not None and final_ans is None) or \
               (calc_result is not None and final_ans is None):
                if proposed not in ("writer", "finish"):
                    self.logger.info("🔄 Принудительно направляем к писателю")
                    proposed = "writer"
            
            self.logger.info(f"🎯 Решение супервайзера: {proposed.upper()}")
            return {
                "next_agent": proposed,
                "iteration": iteration,
                "history_summary": f"Итерация {iteration}: выбран {proposed}",
                "last_agent": proposed
            }
        
        def researcher_node(state: SupervisorState) -> dict:
            start = time.time()
            self.logger.info("🔍 Исследователь начал работу")
            question = state.get("question", "")
            if "RAG" in question.upper():
                facts = "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск информации в базе знаний и генерацию текста. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
            elif "LLM" in question.upper():
                facts = "Большие языковые модели (LLM) обучаются на больших объёмах текстов и способны генерировать связные ответы."
            else:
                facts = "Информация по запросу не найдена. Возможно, требуется уточнить вопрос."
            elapsed = time.time() - start
            self.logger.info(f"🔍 Исследователь завершил за {elapsed:.2f}с, результат: {facts[:50]}...")
            return {
                "research_result": facts,
                "messages": [AIMessage(content=f"📚 РЕЗУЛЬТАТ ИССЛЕДОВАНИЯ:\n{facts}")]
            }
        
        def critic_node(state: SupervisorState) -> dict:
            start = time.time()
            self.logger.info("📊 Критик начал работу")
            research = state.get("research_result", "")
            if "не найдена" in research.lower():
                feedback = "Факты неполные. Рекомендую дополнительный поиск."
            elif len(research) < 30:
                feedback = "Факты слишком краткие. Добавьте детали."
            else:
                feedback = "Факты достаточны и достоверны."
            elapsed = time.time() - start
            self.logger.info(f"📊 Критик завершил за {elapsed:.2f}с, вердикт: {feedback[:50]}...")
            return {
                "critic_feedback": feedback,
                "messages": [AIMessage(content=f"📊 КРИТИК ОДОБРЯЕТ:\n{feedback}")]
            }
        
        def calculator_node(state: SupervisorState) -> dict:
            start = time.time()
            self.logger.info("🧮 Калькулятор начал работу")
            question = state.get("question", "")
            try:
                expr = question.lower()
                result = None
                if '%' in expr and 'от' in expr:
                    parts = expr.split('от')
                    if len(parts) == 2:
                        percent = re.sub(r'[^0-9.]', '', parts[0].strip())
                        number = re.sub(r'[^0-9.]', '', parts[1].strip())
                        if percent and number:
                            result = float(number) * float(percent) / 100
                else:
                    numbers = re.findall(r'[\d.]+', expr)
                    if len(numbers) >= 2:
                        if 'плюс' in expr or '+' in expr:
                            result = sum(float(n) for n in numbers)
                        elif 'минус' in expr or '-' in expr:
                            result = float(numbers[0]) - float(numbers[1])
                        elif 'умножить' in expr or '*' in expr:
                            result = float(numbers[0]) * float(numbers[1])
                        elif 'делить' in expr or '/' in expr:
                            result = float(numbers[0]) / float(numbers[1]) if float(numbers[1]) != 0 else "Ошибка: деление на ноль"
                if result is not None:
                    calc_result = f"Результат вычисления: {result}"
                else:
                    calc_result = "Не удалось распознать математическое выражение. Уточните вопрос."
            except Exception as e:
                calc_result = f"Ошибка вычисления: {e}"
            elapsed = time.time() - start
            self.logger.info(f"🧮 Калькулятор завершил за {elapsed:.2f}с, результат: {calc_result[:50]}...")
            return {
                "calculation_result": calc_result,
                "messages": [AIMessage(content=f"🧮 РЕЗУЛЬТАТ ВЫЧИСЛЕНИЯ:\n{calc_result}")]
            }
        
        def writer_node(state: SupervisorState) -> dict:
            start = time.time()
            self.logger.info("✍️ Писатель начал работу")
            research = state.get("research_result", "")
            feedback = state.get("critic_feedback")
            calc = state.get("calculation_result", "")
            parts = []
            if research:
                parts.append(f"Факты:\n{research}")
            if calc:
                parts.append(f"Вычисления:\n{calc}")
            if feedback:
                parts.append(f"Примечания: {feedback}")
            answer = "\n\n".join(parts) if parts else "Недостаточно данных для ответа."
            elapsed = time.time() - start
            self.logger.info(f"✍️ Писатель завершил за {elapsed:.2f}с, ответ длиной {len(answer)} символов")
            return {
                "final_answer": answer,
                "messages": [AIMessage(content=f"📝 ФИНАЛЬНЫЙ ОТВЕТ:\n{answer}")]
            }
        
        # Маршрутизация
        def route_after_supervisor(state: SupervisorState) -> Literal["researcher", "critic", "writer", "calculator", "finish"]:
            return state.get("next_agent", "finish")
        
        # Сборка графа
        builder = StateGraph(SupervisorState)
        builder.add_node("supervisor", supervisor_node)
        builder.add_node("researcher", researcher_node)
        builder.add_node("critic", critic_node)
        builder.add_node("writer", writer_node)
        builder.add_node("calculator", calculator_node)
        
        builder.set_entry_point("supervisor")
        builder.add_conditional_edges(
            "supervisor",
            route_after_supervisor,
            {
                "researcher": "researcher",
                "critic": "critic",
                "writer": "writer",
                "calculator": "calculator",
                "finish": END
            }
        )
        builder.add_edge("researcher", "supervisor")
        builder.add_edge("critic", "supervisor")
        builder.add_edge("writer", "supervisor")
        builder.add_edge("calculator", "supervisor")
        
        # Чекпоинтер для памяти и human‑in‑the‑loop
        memory = MemorySaver()
        # Если нужен human‑in‑the‑loop, добавляем прерывание перед supervisor
        if self.use_human_loop:
            graph = builder.compile(checkpointer=memory, interrupt_before=["supervisor"])
            self.logger.info("🧑‍💻 Human‑in‑the‑loop включён (прерывание перед supervisor)")
        else:
            graph = builder.compile(checkpointer=memory)
        return graph
    
    # ---- Методы запуска ----
    
    def run(self, question: str, session_id: str = "default", verbose: bool = True) -> str:
        """
        Синхронный запуск системы.
        Если включён human‑in‑the‑loop, запрашивает подтверждение у пользователя.
        """
        self.logger.info(f"📝 Новый вопрос: {question}")
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "question": question,
            "next_agent": "researcher",
            "research_result": None,
            "critic_feedback": None,
            "final_answer": None,
            "calculation_result": None,
            "iteration": 0,
            "history_summary": "Начало работы",
            "error": None,
            "last_agent": None
        }
        config = {"configurable": {"thread_id": session_id}, "recursion_limit": 15}
        
        try:
            if self.use_human_loop:
                # Запускаем до первого прерывания
                result = self.graph.invoke(initial_state, config=config)
                while True:
                    # Проверяем, остановились ли перед supervisor
                    snapshot = self.graph.get_state(config)
                    if snapshot.next == ("supervisor",):
                        # Показываем решение супервайзера
                        proposed = snapshot.values.get("next_agent", "finish")
                        print(f"\n🔮 Супервайзер предлагает вызвать: {proposed.upper()}")
                        confirm = input("✅ Подтвердить? (y/n/e - edit): ").strip().lower()
                        if confirm == 'y':
                            # Продолжаем выполнение
                            self.logger.info(f"✅ Оператор подтвердил: {proposed}")
                            result = self.graph.invoke(None, config=config)
                        elif confirm == 'e':
                            # Редактируем решение вручную
                            new_agent = input("Введите имя агента (researcher/critic/writer/calculator/finish): ").strip()
                            if new_agent in ["researcher", "critic", "writer", "calculator", "finish"]:
                                # Обновляем состояние
                                self.graph.update_state(config, {"next_agent": new_agent})
                                self.logger.info(f"✏️ Оператор изменил на: {new_agent}")
                                result = self.graph.invoke(None, config=config)
                            else:
                                print("❌ Некорректный ввод, пропускаем")
                                result = self.graph.invoke(None, config=config)
                        else:
                            # Отмена – завершаем
                            self.logger.info("❌ Оператор отменил выполнение")
                            return "Выполнение отменено оператором."
                    else:
                        # Если прерывания нет, завершаем
                        break
                # Получаем финальный результат
                final_state = self.graph.get_state(config)
                final_ans = final_state.values.get("final_answer", "Не сгенерирован")
                self.logger.info(f"✅ Финальный ответ получен")
                return final_ans
            else:
                # Обычный запуск без human‑in‑the‑loop
                result = self.graph.invoke(initial_state, config=config)
                final_ans = result.get("final_answer", "Не сгенерирован")
                self.logger.info(f"✅ Финальный ответ получен")
                return final_ans
        except Exception as e:
            self.logger.error(f"❌ Ошибка: {e}", exc_info=True)
            return f"❌ Ошибка: {e}"
    
    async def arun(self, question: str, session_id: str = "default", verbose: bool = True) -> str:
        """Асинхронный запуск (без human‑in‑the‑loop)."""
        self.logger.info(f"📝 Асинхронный вопрос: {question}")
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "question": question,
            "next_agent": "researcher",
            "research_result": None,
            "critic_feedback": None,
            "final_answer": None,
            "calculation_result": None,
            "iteration": 0,
            "history_summary": "Начало работы",
            "error": None,
            "last_agent": None
        }
        config = {"configurable": {"thread_id": session_id}, "recursion_limit": 15}
        try:
            result = await self.graph.ainvoke(initial_state, config=config)
            return result.get("final_answer", "Не сгенерирован")
        except Exception as e:
            self.logger.error(f"❌ Асинхронная ошибка: {e}", exc_info=True)
            return f"❌ Ошибка: {e}"


# ============================================================================
# 5. ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ============================================================================

if __name__ == "__main__":
    # Создаём систему с human‑in‑the‑loop и логированием
    system = SupervisedMultiAgent(
        use_human_loop=True,
        log_file="supervisor_demo.log",
        log_level=logging.INFO
    )
    
    questions = [
        "Что такое RAG?",
        "Сколько будет 25% от 200?",
        "Какая погода в Москве?"
    ]
    
    for q in questions:
        print("\n" + "=" * 60)
        print(f"📝 Вопрос: {q}")
        print("=" * 60)
        answer = system.run(q, session_id="demo_user")
        print(f"\n✅ Финальный ответ:\n{answer}")
        print("-" * 60)
```

---

### 5.5. Как это работает

1. **Логирование** – каждый шаг записывается в файл `supervisor.log` и выводится в консоль. Вы видите, кого выбрал супервайзер, сколько времени работал каждый агент и что они вернули.

2. **Human‑in‑the‑loop** – если `use_human_loop=True`, система останавливается перед каждым решением супервайзера. Оператор видит предложение и может:
   - **y** – подтвердить и продолжить.
   - **e** – изменить агента вручную.
   - **n** – отменить выполнение.

3. **Память** – `MemorySaver` сохраняет состояние между вызовами, поэтому при повторных вопросах в одной сессии агенты помнят историю.

4. **Гибкость** – вы можете легко добавлять новых агентов, просто описав их как инструменты и создав соответствующие узлы.

---

## Заключение Лекции 6.7

Мы построили сложную, но управляемую систему, где супервайзер координирует работу нескольких специализированных агентов. Вот что мы научились делать:

- **Проектировать иерархические multi‑agent системы** с управляющим агентом.
- **Динамически выбирать агентов** в зависимости от задачи.
- **Обеспечивать прозрачность** через логирование и мониторинг.
- **Встраивать человеческий контроль** для безопасного выполнения.
- **Сохранять состояние** между сессиями с помощью чекпоинтеров.

Теперь вы можете создавать системы, где несколько ИИ-агентов работают как слаженная команда под руководством супервайзера. Это уже близко к промышленным решениям, используемым в автоматизации бизнес-процессов, исследованиях и разработке сложных продуктов.

**Что дальше?** Вы можете:
- Добавлять новых агентов (эксперт по данным, редактор, планировщик).
- Интегрировать с реальными API и базами данных.
- Внедрять параллельное выполнение агентов.
- Использовать более мощные модели для супервайзера и агентов.

Поздравляю! Вы освоили один из самых сложных и востребованных паттернов в современной ИИ-разработке.